In [20]:
!pip install -q yfinance langchain-core

In [21]:

import yfinance as yf
from langchain_core.tools import tool

@tool
def get_stock_price(ticker: str) -> str:
    """Fetches the current stock price for a given ticker symbol (e.g. 'AAPL', 'TSLA', 'RELIANCE.NS' for Indian stocks)."""
    try:
        stock = yf.Ticker(ticker)
        info = stock.history(period="1d")
        if info.empty:
            return f"Could not find price data for ticker '{ticker}'. Check if the symbol is correct."

        latest_price = info["Close"].iloc[-1]
        currency = stock.info.get("currency", "")
        return f"The current price of {ticker} is {latest_price:.2f} {currency}"
    except Exception as e:
        return f"Error fetching price for {ticker}: {e}"

# Quick test
print(get_stock_price.invoke({"ticker": "AAPL"}))
print(get_stock_price.invoke({"ticker": "RELIANCE.NS"}))  # Indian stock example (NSE)

The current price of AAPL is 321.66 USD
The current price of RELIANCE.NS is 1272.20 INR


In [24]:
from langchain_groq import ChatGroq
from google.colab import userdata

llm = ChatGroq(
    model="llama-3.3-70b-versatile",
    api_key=userdata.get("GROQ_API_KEY")
)

stock_tools = [get_stock_price]
llm_with_stock_tools = llm.bind_tools(stock_tools)
stock_tool_map = {t.name: t for t in stock_tools}

def handle_stock_request(user_text):
    response = llm_with_stock_tools.invoke([
        ("system", "You are a stock price assistant. Your sole purpose is to fetch the current stock price for a given ticker symbol using the 'get_stock_price' tool. Only call the tool if the user explicitly asks for a stock price. If the user asks anything beyond getting the current stock price (e.g., questions about future growth, news, or analysis), respond that you can only provide current stock prices and cannot answer that specific question."),
        ("human", user_text)
    ])

    if not response.tool_calls:
        print(response.content)
        return

    for tool_call in response.tool_calls:
        tool_name = tool_call["name"]
        tool_args = tool_call["args"]

        if tool_name in stock_tool_map:
            result = stock_tool_map[tool_name].invoke(tool_args)
            print(result)
        else:
            print(f"Tool '{tool_name}' not found")

# ---------------- Try it ----------------
user_input = input("Ask about a stock price: ")
handle_stock_request(user_input)

Ask about a stock price: suggest the comapny i should inverst in 
I can only provide current stock prices and cannot answer questions about investment suggestions or future growth. If you would like to know the current stock price of a specific company, please let me know the ticker symbol and I can help you with that.
